In [10]:
# CELL 1: SETUP ENVIRONMENT

import os
import shutil
import subprocess
import sys
import time

# Periksa Java di runtime
if shutil.which("java") is None:
    subprocess.run(
        ["bash", "-lc",
         "apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless"],
        check=True
    )

# Atur JAVA_HOME
java_path = shutil.which("java")
java_home = os.path.dirname(
    os.path.dirname(os.path.realpath(java_path))
)
os.environ["JAVA_HOME"] = java_home

print("Python:", sys.version.split()[0])
print("Java:", subprocess.check_output(
    ["java", "-version"], stderr=subprocess.STDOUT
).decode().splitlines()[0])

# Instal PySpark
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pyspark"],
    check=True
)

print("Setup selesai.")


Python: 3.13.16
Java: [0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Setup selesai.


In [11]:
# CELL 2: SPARK SESSION

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("BigDataBiopsyAnalytics")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext

print("Spark version :", spark.version)
print("Execution mode:", sc.master)
print("Default parallelism:", sc.defaultParallelism)
print("Shuffle partitions:",
      spark.conf.get("spark.sql.shuffle.partitions"))


Spark version : 4.0.4
Execution mode: local[*]
Default parallelism: 2
Shuffle partitions: 8


In [12]:
# CELL 3: GENERATE SYNTHETIC BIG DATA

from pyspark.sql import functions as F

# Target awal: 1 juta baris
N_RECORDS = 1_000_000

# Jumlah partisi awal; sesuaikan dengan kapasitas runtime
N_PARTITIONS = max(8, sc.defaultParallelism * 2)

base_df = spark.range(
    start=0,
    end=N_RECORDS,
    numPartitions=N_PARTITIONS
).withColumnRenamed("id", "patient_id")

# Membuat metadata sintetis secara deterministik
biopsy_df = (
    base_df
    .withColumn(
        "age",
        (F.pmod(F.col("patient_id") * 17, F.lit(63)) + 18)
        .cast("int")
    )
    .withColumn(
        "tissue_type",
        F.when(F.pmod(F.col("patient_id"), 3) == 0, "breast")
         .when(F.pmod(F.col("patient_id"), 3) == 1, "lung")
         .otherwise("skin")
    )
    .withColumn(
        "finding",
        F.when(F.pmod(F.col("patient_id"), 10) < 7, "normal")
         .when(F.pmod(F.col("patient_id"), 10) < 9, "abnormal")
         .otherwise("suspicious")
    )
    .withColumn(
        "mitotic_count",
        F.pmod(F.col("patient_id") * 7, 12).cast("int")
    )
    .withColumn(
        "pathology_note",
        F.concat_ws(
            " ",
            F.col("finding"),
            F.lit("tissue sample"),
            F.when(
                F.col("mitotic_count") >= 6,
                F.lit("high mitotic activity")
            ).otherwise(F.lit("low mitotic activity")),
            F.lit("pathology review")
        )
    )
)

# Simpan dalam cache untuk dipakai beberapa kali
biopsy_df = biopsy_df.cache()

# Materialisasi cache dan verifikasi jumlah baris
start = time.perf_counter()
total_records = biopsy_df.count()
generation_time = time.perf_counter() - start

print(f"Total records : {total_records:,}")
print(f"Partitions    : {biopsy_df.rdd.getNumPartitions()}")
print(f"Count time    : {generation_time:.2f} seconds")

biopsy_df.show(10, truncate=False)
biopsy_df.printSchema()


Total records : 1,000,000
Partitions    : 8
Count time    : 0.62 seconds
+----------+---+-----------+----------+-------------+--------------------------------------------------------------+
|patient_id|age|tissue_type|finding   |mitotic_count|pathology_note                                                |
+----------+---+-----------+----------+-------------+--------------------------------------------------------------+
|0         |18 |breast     |normal    |0            |normal tissue sample low mitotic activity pathology review    |
|1         |35 |lung       |normal    |7            |normal tissue sample high mitotic activity pathology review   |
|2         |52 |skin       |normal    |2            |normal tissue sample low mitotic activity pathology review    |
|3         |69 |breast     |normal    |9            |normal tissue sample high mitotic activity pathology review   |
|4         |23 |lung       |normal    |4            |normal tissue sample low mitotic activity pathology rev

In [13]:
# CELL 4: DISTRIBUTED WORD COUNT WITH RDD

# Mengambil kolom catatan sebagai RDD
notes_rdd = (
    biopsy_df
    .select("pathology_note")
    .rdd
    .map(lambda row: row["pathology_note"])
)

# Transformation: setiap catatan dipecah menjadi kata
words_rdd = notes_rdd.flatMap(
    lambda note: note.lower().split()
)

# Transformation: setiap kata menjadi pasangan (kata, 1)
pairs_rdd = words_rdd.map(
    lambda word: (word, 1)
)

# Aggregation: hitung frekuensi kata pada seluruh partisi
word_counts_rdd = pairs_rdd.reduceByKey(
    lambda a, b: a + b,
    numPartitions=N_PARTITIONS
)

# Action: memicu eksekusi dan mengumpulkan 15 hasil teratas
start = time.perf_counter()

top_words = word_counts_rdd.takeOrdered(
    15,
    key=lambda item: (-item[1], item[0])
)

rdd_wordcount_time = time.perf_counter() - start

print("=== TOP 15 WORD FREQUENCIES ===")
for word, count in top_words:
    print(f"{word:25s} {count:>10,}")

print(f"\nWord count time: {rdd_wordcount_time:.2f} seconds")
print("RDD partitions:", words_rdd.getNumPartitions())


=== TOP 15 WORD FREQUENCIES ===
activity                   1,000,000
mitotic                    1,000,000
pathology                  1,000,000
review                     1,000,000
sample                     1,000,000
tissue                     1,000,000
normal                       700,000
high                         500,000
low                          500,000
abnormal                     200,000
suspicious                   100,000

Word count time: 18.99 seconds
RDD partitions: 8


In [14]:
# CELL 5: DATAFRAME ANALYTICS

from pyspark.sql import functions as F

# Aturan demonstrasi, bukan kriteria diagnosis klinis
analyzed_df = biopsy_df.withColumn(
    "review_status",
    F.when(
        (F.col("finding").isin("abnormal", "suspicious"))
        | (F.col("mitotic_count") >= 6),
        "Needs Further Review"
    ).otherwise("No Rule-Based Flag")
)

print("=== SUMMARY BY REVIEW STATUS ===")

status_summary = (
    analyzed_df
    .groupBy("review_status")
    .agg(
        F.count("*").alias("record_count"),
        F.avg("age").alias("average_age"),
        F.avg("mitotic_count").alias("average_mitotic_count")
    )
    .orderBy("review_status")
)

status_summary.show(truncate=False)

print("=== SUMMARY BY TISSUE TYPE ===")

tissue_summary = (
    analyzed_df
    .groupBy("tissue_type", "finding")
    .agg(
        F.count("*").alias("record_count"),
        F.avg("mitotic_count").alias("average_mitotic_count")
    )
    .orderBy("tissue_type", "finding")
)

tissue_summary.show(truncate=False)

print("=== AGE GROUP ANALYSIS ===")

age_summary = (
    analyzed_df
    .withColumn(
        "age_group",
        F.when(F.col("age") < 30, "18-29")
         .when(F.col("age") < 45, "30-44")
         .when(F.col("age") < 60, "45-59")
         .otherwise("60+")
    )
    .groupBy("age_group", "review_status")
    .count()
    .orderBy("age_group", "review_status")
)

age_summary.show(truncate=False)


=== SUMMARY BY REVIEW STATUS ===
+--------------------+------------+-----------------+---------------------+
|review_status       |record_count|average_age      |average_mitotic_count|
+--------------------+------------+-----------------+---------------------+
|Needs Further Review|649999      |48.99993384605207|7.1538479289968135   |
|No Rule-Based Flag  |350001      |49.00003428561633|2.428570204085131    |
+--------------------+------------+-----------------+---------------------+

=== SUMMARY BY TISSUE TYPE ===
+-----------+----------+------------+---------------------+
|tissue_type|finding   |record_count|average_mitotic_count|
+-----------+----------+------------+---------------------+
|breast     |abnormal  |66666       |4.500090000900009    |
|breast     |normal    |233334      |4.285684898043148    |
|breast     |suspicious|33334       |6.0                  |
|lung       |abnormal  |66667       |5.499932500337498    |
|lung       |normal    |233333      |5.71432244903207     |

In [15]:
# CELL 6: PARTITION BENCHMARK

partition_results = []

for shuffle_parts in [2, 4, 8, 16]:
    spark.conf.set(
        "spark.sql.shuffle.partitions",
        str(shuffle_parts)
    )

    start = time.perf_counter()

    result = (
        biopsy_df
        .groupBy("tissue_type", "finding")
        .agg(F.count("*").alias("total"))
    )

    # Action untuk menjalankan pekerjaan
    result.collect()

    elapsed = time.perf_counter() - start

    partition_results.append(
        (shuffle_parts, round(elapsed, 3))
    )

print("=== SHUFFLE PARTITION BENCHMARK ===")
print("Partitions | Execution time (seconds)")

for parts, elapsed in partition_results:
    print(f"{parts:10d} | {elapsed:8.3f}")


=== SHUFFLE PARTITION BENCHMARK ===
Partitions | Execution time (seconds)
         2 |    0.644
         4 |    0.579
         8 |    0.765
        16 |    0.565


In [16]:
# CELL 7: INSPECT EXECUTION PLAN

print("=== DATAFRAME EXECUTION PLAN ===")

(
    biopsy_df
    .groupBy("tissue_type", "finding")
    .count()
    .explain(mode="formatted")
)

print("=== RDD LINEAGE ===")
print(word_counts_rdd.toDebugString().decode("utf-8"))


=== DATAFRAME EXECUTION PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (9)
+- HashAggregate (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- InMemoryTableScan (1)
               +- InMemoryRelation (2)
                     +- * Project (5)
                        +- * Project (4)
                           +- * Range (3)


(1) InMemoryTableScan
Output [2]: [tissue_type#2089, finding#2090]
Arguments: [tissue_type#2089, finding#2090]

(2) InMemoryRelation
Arguments: [patient_id#2087L, age#2088, tissue_type#2089, finding#2090, mitotic_count#2091, pathology_note#2092], StorageLevel(disk, memory, deserialized, 1 replicas), [patient_id#2087L ASC NULLS FIRST]

(3) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(8))

(4) Project [codegen id : 1]
Output [5]: [id#0L AS patient_id#1L, cast((pmod((id#0L * 17), 63) + 18) as int) AS age#2, CASE WHEN (pmod(id#0L, 3) = 0) THEN breast WHEN (pmod(id#0L, 3) = 1) THEN lung ELSE skin END AS tissue_t

In [17]:
# CELL 8: DATA-SCALE BENCHMARK

scale_results = []

for n in [100_000, 500_000, 1_000_000]:
    test_df = (
        spark.range(0, n, numPartitions=N_PARTITIONS)
        .withColumn(
            "tissue_type",
            F.when(F.pmod(F.col("id"), 3) == 0, "breast")
             .when(F.pmod(F.col("id"), 3) == 1, "lung")
             .otherwise("skin")
        )
        .withColumn(
            "finding",
            F.when(F.pmod(F.col("id"), 10) < 7, "normal")
             .when(F.pmod(F.col("id"), 10) < 9, "abnormal")
             .otherwise("suspicious")
        )
    )

    start = time.perf_counter()

    summary = (
        test_df
        .groupBy("tissue_type", "finding")
        .count()
        .collect()
    )

    elapsed = time.perf_counter() - start

    scale_results.append({
        "records": n,
        "seconds": round(elapsed, 3),
        "groups": len(summary)
    })

print("=== DATA SCALE BENCHMARK ===")

for item in scale_results:
    print(
        f"Records: {item['records']:>9,} | "
        f"Time: {item['seconds']:>8.3f} s | "
        f"Groups: {item['groups']}"
    )


=== DATA SCALE BENCHMARK ===
Records:   100,000 | Time:    0.279 s | Groups: 9
Records:   500,000 | Time:    0.311 s | Groups: 9
Records: 1,000,000 | Time:    0.337 s | Groups: 9


In [18]:
# CELL 9: SAVE AGGREGATED RESULTS

output_path = "/content/biopsy_summary_parquet"

(
    tissue_summary
    .write
    .mode("overwrite")
    .parquet(output_path)
)

print("Ringkasan tersimpan di:", output_path)

# Baca kembali untuk memverifikasi
saved_summary = spark.read.parquet(output_path)
saved_summary.show(truncate=False)


Ringkasan tersimpan di: /content/biopsy_summary_parquet
+-----------+----------+------------+---------------------+
|tissue_type|finding   |record_count|average_mitotic_count|
+-----------+----------+------------+---------------------+
|breast     |abnormal  |66666       |4.500090000900009    |
|breast     |normal    |233334      |4.285684898043148    |
|breast     |suspicious|33334       |6.0                  |
|lung       |abnormal  |66667       |5.499932500337498    |
|lung       |normal    |233333      |5.71432244903207     |
|lung       |suspicious|33333       |3.999909999099991    |
|skin       |abnormal  |66667       |6.500022499887501    |
|skin       |normal    |233333      |6.28567755096793     |
|skin       |suspicious|33333       |8.000090000900009    |
+-----------+----------+------------+---------------------+

